# Insurance LLM Fine-Tuning (Kaggle Edition)

**Model:** Qwen2.5-3B-Instruct + QLoRA 4-bit  
**Pipeline:** SFT → DPO → Evaluation  
**GPU:** T4/P100 (Kaggle free)  
**Time:** ~40 min total  

⚠️ **Her aşama sonrası dosyalar `/kaggle/working/` altına kaydedilir — notebook kapansa bile erişilebilir.**

## Step 0: GPU Check & Setup

In [1]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("GPU not available! Enable GPU: Settings → Accelerator → GPU T4x2 or P100")

Fri Aug 14 11:11:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# === HF HUB LOGIN (kalıcı depolama) ===
from huggingface_hub import login, HfApi

# Kaggle'da: Settings → Secrets → Add → Name: HF_TOKEN, Value: hf_xxxxx
# Token al: https://huggingface.co/settings/tokens (Write permission!)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")  # ← kendi tokenını yaz
login(token=HF_TOKEN)

# Repo adları (kendi username'ini yaz)
HF_USER = "sefabilicier"  # ← BUNU DEĞİŞTİR
SFT_REPO = f"{HF_USER}/insurance-qwen3b-sft"
DPO_REPO = f"{HF_USER}/insurance-qwen3b-dpo"
EVAL_REPO = f"{HF_USER}/insurance-llm-eval-results"

api = HfApi()

# Repoları oluştur (yoksa)
for repo in [SFT_REPO, DPO_REPO, EVAL_REPO]:
    api.create_repo(repo, exist_ok=True, private=True)

print(f"✅ HF Hub ready")
print(f"   SFT  → {SFT_REPO}")
print(f"   DPO  → {DPO_REPO}")
print(f"   Eval → {EVAL_REPO}")

✅ HF Hub ready
   SFT  → sefabilicier/insurance-qwen3b-sft
   DPO  → sefabilicier/insurance-qwen3b-dpo
   Eval → sefabilicier/insurance-llm-eval-results


In [2]:
!pip install -q \
    huggingface_hub==0.34.1 \
    peft==0.16.0 \
    transformers==4.51.3 \
    trl==0.16.1 \
    accelerate==1.6.0 \
    bitsandbytes>=0.46.1

## Step 1: Upload Project

**Seçenek A:** Kaggle Dataset olarak yükle (önerilen, kalıcı)  
**Seçenek B:** Zip upload  
**Seçenek C:** GitHub clone

In [3]:
from pathlib import Path
import os

PROJECT_DIR = Path("/kaggle/input/datasets/sefabilicier/llm-insurance-finetuning")

os.chdir(PROJECT_DIR)

print(os.getcwd())

/kaggle/input/datasets/sefabilicier/llm-insurance-finetuning


## Step 2: Generate Data (if needed)

In [4]:
import json
from pathlib import Path

if Path('./data/splits/train.json').exists():
    train_data = json.load(open('./data/splits/train.json'))
    val_data = json.load(open('./data/splits/validation.json'))
    test_data = json.load(open('./data/splits/test.json'))
    print(f"Data exists: Train={len(train_data)} Val={len(val_data)} Test={len(test_data)}")
else:
    print("Generating data...")
    !python scripts/prepare_data.py --step all --num-examples 1000 --template-ratio 1.0
    train_data = json.load(open('./data/splits/train.json'))
    val_data = json.load(open('./data/splits/validation.json'))
    test_data = json.load(open('./data/splits/test.json'))
    print(f"Generated: Train={len(train_data)} Val={len(val_data)} Test={len(test_data)}")

Data exists: Train=797 Val=99 Test=101


In [5]:
import requests

try:
    r = requests.get("https://huggingface.co", timeout=10)
    print(r.status_code)
except Exception as e:
    print(e)

200


## Step 3: Load Model + QLoRA

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# LoRA
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded | Trainable: {trainable:,} ({trainable/total*100:.2f}%) | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB")

Loading Qwen/Qwen2.5-3B-Instruct...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Model loaded | Trainable: 29,933,568 (1.73%) | VRAM: 1.4GB


## Step 4: Prepare Datasets

In [7]:
from datasets import Dataset
import random

SYSTEM_PROMPT = """You are an expert insurance support agent for a Turkish insurance company.

You help customers with:
- Policy inquiries and explanations
- Claims processing guidance
- Coverage questions
- Premium and billing information
- Policy modifications and renewals

Respond professionally, accurately, and within company policies. Always be helpful and clear."""

def format_chatml(example):
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n{example['user']}\n<|im_end|>\n"
        f"<|im_start|>assistant\n{example['assistant']}\n<|im_end|>"
    )
    return {"text": text}

# SFT datasets
train_dataset = Dataset.from_list(train_data).map(format_chatml)
val_dataset = Dataset.from_list(val_data).map(format_chatml)

# DPO preference pairs
REJECTIONS = [
    "I'm not sure about that. You should check your policy documents or call us back later.",
    "Yes, that should be covered. Let me know if you have other questions.",
    "Just send us an email about it and we'll figure it out eventually.",
    "Look, I don't know the details of your policy off the top of my head. Check it yourself.",
    "Don't worry, everything is definitely covered. We'll take care of everything no matter what.",
    "Please check your policy.",
    "Our policies vary. I recommend reviewing your specific policy documentation for details.",
    "That's handled by a different department. Call them during business hours.",
]

def build_pref_pairs(examples, seed=42):
    rng = random.Random(seed)
    pairs = []
    for ex in examples:
        prompt = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
            f"<|im_start|>user\n{ex['user']}\n<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        pairs.append({
            "prompt": prompt,
            "chosen": ex['assistant'] + "\n<|im_end|>",
            "rejected": rng.choice(REJECTIONS) + "\n<|im_end|>",
        })
    return pairs

train_pref = Dataset.from_list(build_pref_pairs(train_data, 42))
val_pref = Dataset.from_list(build_pref_pairs(val_data, 43))

print(f"SFT  → Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"DPO  → Train: {len(train_pref)} | Val: {len(val_pref)}")

Map:   0%|          | 0/797 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

SFT  → Train: 797 | Val: 99
DPO  → Train: 797 | Val: 99


---
## Step 5: SFT Training 🚀

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

sft_config = SFTConfig(
    output_dir="/kaggle/working/outputs/checkpoints/sft",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    max_seq_length=1024,
    packing=False,
    dataset_text_field="text",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    report_to="none",
    seed=42,
    # === HF HUB AUTO-PUSH ===
    push_to_hub=True,
    hub_model_id=SFT_REPO,
    hub_token=HF_TOKEN,
    hub_strategy="checkpoint",      # Her checkpoint'ta push
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting SFT...")
sft_result = sft_trainer.train()

# === Training biter bitmez push (ekstra güvenlik) ===
sft_trainer.push_to_hub(commit_message="SFT training complete")
print(f"\n✅ SFT Done | Loss: {sft_result.metrics.get('train_loss', 'N/A'):.4f}")
print(f"✅ Pushed to: https://huggingface.co/{SFT_REPO}")

In [8]:
!find /kaggle/working/outputs -name "adapter_config.json" 2>/dev/null

### IF NEEDED
If we need to work again, do not run sft again.

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login, snapshot_download
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
)

# Download then load locally
SFT_LOCAL = "/kaggle/working/sft_adapter"
snapshot_download(repo_id="sefabilicier/insurance-qwen3b-sft", local_dir=SFT_LOCAL, token=HF_TOKEN)

model = PeftModel.from_pretrained(base_model, SFT_LOCAL, is_trainable=True)
tokenizer = AutoTokenizer.from_pretrained(SFT_LOCAL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Ready for DPO | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

last-checkpoint/adapter_model.safetensor(…):   0%|          | 0.00/120M [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/120M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/6.03k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

✅ Ready for DPO | VRAM: 2.3GB


---
## Step 6: DPO Training 🚀

In [16]:
from trl import DPOTrainer, DPOConfig

torch.cuda.empty_cache()

dpo_config = DPOConfig(
    output_dir="/kaggle/working/outputs/checkpoints/dpo",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    beta=0.1,
    loss_type="sigmoid",
    max_length=1024,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=25,
    report_to="none",
    seed=42,
    # === HF HUB AUTO-PUSH ===
    push_to_hub=True,
    hub_model_id=DPO_REPO,
    hub_token=HF_TOKEN,
    hub_strategy="checkpoint",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=train_pref,
    eval_dataset=val_pref,
    processing_class=tokenizer,
)

print("Starting DPO...")
dpo_result = dpo_trainer.train()

dpo_trainer.push_to_hub(commit_message="DPO training complete")
print(f"\n✅ DPO Done | Loss: {dpo_result.metrics.get('train_loss', 'N/A'):.4f}")
print(f"✅ Pushed to: https://huggingface.co/{DPO_REPO}")

Extracting prompt from train dataset:   0%|          | 0/797 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/797 [00:00<?, ? examples/s]

Extracting prompt from eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/797 [00:00<?, ? examples/s]

...

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream 

Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
50,0.000000,0.000000,18.405815,-12.458596,1.000000,30.864412,-23.609709,-242.085312,-4.339113,-3.726041
100,0.000000,0.000000,18.868803,-14.008465,1.000000,32.877266,-18.979813,-257.583984,-4.343635,-3.824076
150,0.000000,0.000000,18.891619,-14.001675,1.000000,32.893295,-18.751667,-257.516083,-4.343133,-3.823314


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Uploading...:   0%|          | 0.00/131M [00:00<?, ?B/s]


✅ DPO Done | Loss: 0.0000
✅ Pushed to: https://huggingface.co/sefabilicier/insurance-qwen3b-dpo


In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login, snapshot_download
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
)

# DPO adapter: Hub → local → load
DPO_LOCAL = "/kaggle/working/dpo_adapter"
snapshot_download(repo_id="sefabilicier/insurance-qwen3b-dpo", local_dir=DPO_LOCAL, token=HF_TOKEN)

model = PeftModel.from_pretrained(base_model, DPO_LOCAL, is_trainable=False)
tokenizer = AutoTokenizer.from_pretrained(DPO_LOCAL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ DPO model loaded | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/120M [00:00<?, ?B/s]

last-checkpoint/adapter_model.safetensor(…):   0%|          | 0.00/120M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

training_args.bin:   0%|          | 0.00/6.61k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

✅ DPO model loaded | VRAM: 3.2GB


---
## Step 7: Evaluation

In [12]:
torch.cuda.empty_cache()
model.eval()

SYSTEM_PROMPT = """You are an expert insurance support agent for a Turkish insurance company.

You help customers with:
- Policy inquiries and explanations
- Claims processing guidance
- Coverage questions
- Premium and billing information
- Policy modifications and renewals

Respond professionally, accurately, and within company policies. Always be helpful and clear."""

test_questions = [
    "What is my deductible on my auto insurance policy?",
    "I was in a car accident. How do I file a claim?",
    "Does my policy cover roadside assistance?",
    "Why did my premium increase this year?",
    "I want to add my spouse to my policy.",
]

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)

for q in test_questions:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n{q}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    print(f"\nQ: {q}")
    print(f"A: {resp[:300]}")
    print("-" * 40)

INFERENCE TEST

Q: What is my deductible on my auto insurance policy?
A: I understand your concern. Your auto insurance deductible is currently set at $2,500. This is the amount you would need to pay out-of-pocket before your insurance coverage kicks in for any covered claim. Let me know if you have any other concerns.
----------------------------------------

Q: I was in a car accident. How do I file a claim?
A: Thank you for reaching out to us. I'm sorry to hear about the incident. To file a claim for the car accident, please follow these steps: 1) Gather all relevant documentation including photos, police reports, and witness information. 2) Submit your claim through our online portal or call our claims h
----------------------------------------

Q: Does my policy cover roadside assistance?
A: I understand your concern. Yes, your policy does include roadside assistance coverage. Your policy provides comprehensive protection up to $250,000 for such incidents. Please note that there m

In [14]:
import os, json, sys

# Proje kodlarına erişim (read-only input'tan)
sys.path.insert(0, '/kaggle/input/datasets/sefabilicier/llm-insurance-finetuning')

from src.evaluation.metrics import compute_task_metrics

# Test verisini yükle
test_data = json.load(open('/kaggle/input/datasets/sefabilicier/llm-insurance-finetuning/data/splits/test.json'))
references = [ex['assistant'] for ex in test_data]
categories = [ex.get('category', 'unknown') for ex in test_data]

# Generate predictions
predictions = []
for i, ex in enumerate(test_data):
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n{ex['user']}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    predictions.append(resp)
    if (i+1) % 20 == 0:
        print(f"  Inference: {i+1}/{len(test_data)}")

# Compute metrics
metrics = compute_task_metrics(predictions, references, categories)

print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
for k, v in metrics['overall'].items():
    print(f"  {k:20s}: {v:.4f}" if isinstance(v, float) else f"  {k:20s}: {v}")

print("\nPer category:")
for cat, m in sorted(metrics['per_category'].items()):
    print(f"  {cat:25s} ROUGE-L:{m['rouge_l']:.4f} KW:{m['keyword_coverage']:.4f} FMT:{m['format_compliance']:.4f}")

# Save to writable directory
EVAL_DIR = "/kaggle/working/evaluation"
os.makedirs(EVAL_DIR, exist_ok=True)

with open(f'{EVAL_DIR}/final_results.json', 'w') as f:
    json.dump(metrics, f, indent=2)

eval_details = []
for ex, pred, ref in zip(test_data, predictions, references):
    eval_details.append({
        "user": ex["user"],
        "category": ex.get("category", ""),
        "reference": ref,
        "prediction": pred,
    })
with open(f'{EVAL_DIR}/predictions.json', 'w') as f:
    json.dump(eval_details, f, indent=2, ensure_ascii=False)

print(f"\n✅ Results saved to {EVAL_DIR}/")

  Inference: 20/101
  Inference: 40/101
  Inference: 60/101
  Inference: 80/101
  Inference: 100/101

EVALUATION RESULTS
  rouge_1             : 0.7690
  rouge_2             : 0.6974
  rouge_l             : 0.7544
  bleu                : 0.6665
  keyword_coverage    : 0.8911
  format_compliance   : 1.0000
  num_examples        : 101

Per category:
  claim_processing          ROUGE-L:0.7887 KW:1.0000 FMT:1.0000
  coverage_questions        ROUGE-L:0.7773 KW:0.8947 FMT:1.0000
  policy_inquiry            ROUGE-L:0.6703 KW:0.9200 FMT:1.0000
  policy_modifications      ROUGE-L:0.7986 KW:0.6905 FMT:1.0000
  premium_billing           ROUGE-L:0.7615 KW:0.9722 FMT:1.0000

✅ Results saved to /kaggle/working/evaluation/


In [19]:
from huggingface_hub import HfApi

api = HfApi()
EVAL_REPO = "sefabilicier/insurance-llm-eval-results"
EVAL_DIR = "/kaggle/working/evaluation/"

api.create_repo(EVAL_REPO, exist_ok=True, private=True, token=HF_TOKEN)

api.upload_file(
    path_or_fileobj=f"{EVAL_DIR}/final_results.json",
    path_in_repo="final_results.json",
    repo_id=EVAL_REPO,
    token=HF_TOKEN,
)
api.upload_file(
    path_or_fileobj=f"{EVAL_DIR}/predictions.json",
    path_in_repo="predictions.json",
    repo_id=EVAL_REPO,
    token=HF_TOKEN,
)
print(f"✅ Eval results pushed to: https://huggingface.co/{EVAL_REPO}")

✅ Eval results pushed to: https://huggingface.co/sefabilicier/insurance-llm-eval-results


---
## Step 8: Download Everything

In [22]:
from IPython.display import FileLink, display
import os

!zip -r /kaggle/working/training_results.zip \
    /kaggle/working/sft_adapter/ \
    /kaggle/working/dpo_adapter/ \
    /kaggle/working/outputs/evaluation/ \
    2>/dev/null

print("\n✅ All results zipped: /kaggle/working/training_results.zip")
!ls -lh /kaggle/working/training_results.zip

print("\n📥 **Dosyayı doğrudan indirmek için tıkla:**")
display(FileLink('/kaggle/working/training_results.zip'))


print("\n📥 Download: Kaggle → Output tab → training_results.zip")
print("📦 HF Hub'da da kalıcı:")
print("   SFT → huggingface.co/sefabilicier/insurance-qwen3b-sft")
print("   DPO → huggingface.co/sefabilicier/insurance-qwen3b-dpo")
print("   Eval → huggingface.co/sefabilicier/insurance-llm-eval-results")

	zip warning: name not matched: /kaggle/working/outputs/evaluation/
updating: kaggle/working/sft_adapter/ (stored 0%)
updating: kaggle/working/sft_adapter/last-checkpoint/ (stored 0%)
updating: kaggle/working/sft_adapter/last-checkpoint/vocab.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 61%)
updating: kaggle/working/sft_adapter/last-checkpoint/trainer_state.json (deflated 68%)
updating: kaggle/working/sft_adapter/last-checkpoint/added_tokens.json (deflated 67%)
updating: kaggle/working/sft_adapter/last-checkpoint/README.md (deflated 66%)
updating: kaggle/working/sft_adapter/last-checkpoint/scheduler.pt (deflated 62%)
updating: kaggle/working/sft_adapter/last-checkpoint/training_args.bin (deflated 52%)
updating: kaggle/working/sft_adapter/last-checkpoint/tokenizer.json (deflated 81%)
updating: kaggle/working/sft_adapter/last-checkpoint/rng_state.pth (deflated 26%)
updating: kaggle/working/sft_adapter/last-checkpoint/adapter_config.json (deflated 54%)
updating: kaggle/working/sft_adapter/last-checkpoint/optimizer.pt (deflated 8%)
updating: kaggle/working/sft_adapter/last-checkpoint/merges.txt (deflated 57%)
updating: kaggle/working/sft_adapter/last-checkpoint/special_tokens_map.json (deflated 69%)
updating: kaggle/working/sft_adapter/last-checkpoint/adapter_m

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/kaggle/working/training_results.zip


📥 Download: Kaggle → Output tab → training_results.zip
📦 HF Hub'da da kalıcı:
   SFT → huggingface.co/sefabilicier/insurance-qwen3b-sft
   DPO → huggingface.co/sefabilicier/insurance-qwen3b-dpo
   Eval → huggingface.co/sefabilicier/insurance-llm-eval-results


## Training Summary

| Phase | Model | Batch | Epochs | Est. Time |
|---|---|---|---|---|
| SFT | Qwen2.5-3B + QLoRA 4-bit | 2×4=8 | 3 | ~20 min |
| DPO | SFT model + QLoRA | 1×4=4 | 1 | ~10 min |
| Eval | Inference on test set | 1 | — | ~5 min |

**Outputs:**
- `sft_adapter_backup/` — SFT LoRA weights
- `dpo_adapter_backup/` — DPO LoRA weights  
- `outputs/evaluation/final_results.json` — Metrics
- `outputs/evaluation/predictions.json` — All predictions vs references
- `training_results.zip` — Everything zipped